# 01a — Fixed-log: Data Labelling

## Setup

Imports → connect → open session. See `01_labelling_guide.ipynb` for full session lifecycle rules.

In [ ]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
# Run once at the start of every session (fresh or resumed).
import json, sys, os


# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

from amygda_ops_risk_score import OpsRiskClient, SessionConfig
from amygda_ops_risk_score import helpers

# Check SDK version
import amygda_ops_risk_score
print('amygda_ops_risk_score version:', amygda_ops_risk_score.__version__)

print("SDK imported successfully.")

In [ ]:
# ── Cell 2a: Connect and wait for the API ────────────────────────────────────
# Run this every time you start a fresh session.
#
# The API runs on a cloud service with scale-to-zero.
# wait_until_ready() polls until all ML models are loaded (cold start: 60-120 s).
# On a warm instance it returns immediately.

client = OpsRiskClient() 
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Your API key and artifact directory ─────────────────────────────────────
# Get your API key from portal.amygda.io → API Keys → Create Key

API_KEY      = "xxx"   # ← paste your key here
ARTIFACT_DIR = "artifacts/fixed_log/labelling/"

import os
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f"Artifact dir ready: {ARTIFACT_DIR}")

In [ ]:
# ── Cell 2c: Open a session ──────────────────────────────────────────────────
# open_session() creates a new session — or auto-resumes the existing one
# if a valid session already exists for this api_key.
# Only one session per api_key is allowed at a time.

config  = SessionConfig(name="my-labelling-run")   # ← give it a meaningful name
session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)

AUTH_ID = session.auth_id
print(f"AUTH_ID  : {AUTH_ID}")
print("Save AUTH_ID — you will need it if your kernel restarts.")
print()
print(json.dumps(session.status(), indent=2))

In [ ]:
# # ── Cell 3: Kernel restarted — reconnect to your existing session ─────────────
# # Use this cell (instead of Cell 2) when your kernel restarted but the session
# # on the server is still alive.
# #
# # 1. Paste the AUTH_ID that was printed when you ran Cell 2c.
# # 2. Set ARTIFACT_DIR to the same path you used in Cell 2b.
# # 3. Run this cell. get_session() makes one status check — no re-registration needed.
# #
# # ─────────────────────────────────────────────────────────────────────────────

# AUTH_ID      = "ses-xxxx"   # ← paste your saved AUTH_ID
# API_KEY      = "xxxx"        # ← paste your saved API_KEY
# ARTIFACT_DIR = "artifacts/fixed_log/labelling/"           # ← same path as in Cell 2b

# client  = OpsRiskClient()
# client.wait_until_ready()

# session = client.get_session(AUTH_ID, artifact_dir=ARTIFACT_DIR)

# status = session.status()
# print(json.dumps(status, indent=2))
# print()
# print("Step state guide:")
# print("  DONE    → completed successfully — skip this step")
# print("  RUNNING → step is in-flight on the server.")
# print("            Call session.status() again in a moment to check if it finished.")
# print("  FAILED  → step failed — safe to re-run")
# print("  NONE    → not started — proceed normally")
# print()
# print("To restore a lost variable from an artifact, e.g.:")
# print(f"  with open("{ARTIFACT_DIR}generate_hierarchy.json") as f:")
# print(f"      hierarchy_result = json.load(f)")
# print()
# print("If get_session() raised SessionNotFoundError, the TTL has expired.")
# print("Run Cell 2a → 2b → 2c to open a fresh session (no re-registration needed).")

In [ ]:
# ── Network outage recovery — run only if session.status() raises APIError(404) ──
# If your network dropped and came back, your in-memory session object is fine.
# Just continue — the SDK reconnects on the next call.
#
# If session.status() raises APIError (404), the session expired during the outage.
# Run the cell below to reattach — no re-registration needed.
#
# client.open_session() auto-resumes if the session is still alive;
# otherwise it creates a fresh one with the same api_key.

# session = client.open_session(api_key=API_KEY, config=config, artifact_dir=ARTIFACT_DIR)
# print(json.dumps(session.status(), indent=2))

# ── Step locked? Delete and recreate the session to start over ────────────────
# Steps lock once a downstream step runs.  restart_session() wipes all server-side
# data and opens a fresh session — api_key is never affected.
#
# session = client.restart_session(session, api_key=API_KEY, config=config)
# print("Session restarted — re-run from Step 1.")

## Step 1 — configure_labelling_pipeline

Upload your log file and lock in pipeline config. See `01_labelling_guide.ipynb` for details.

In [ ]:
# ── Step 1 params — edit here ─────────────────────────────────────────────────
FILE_PATH       = "sample_data/fixed_log_training.csv"            # path to your .csv / .xlsx / .xls file
LOG_COLUMN      = "event_code"                   # column that contains the log text
ASSET_CONTEXT   = "automated sorting station"    # plain-English description of the asset type
MAX_SYSTEMS     = 5                          # how many top-level systems to generate (1–10)
MAX_SUBSYSTEMS  = 4                          # subsystems per system (1–5)
IS_FREE_TEXT    = False                       # True = unstructured free text, False = structured codes
SHEET_NAME      = None                      # XLSX only — None = auto-detect single sheet

configure_result = session.configure_labelling_pipeline(
    file_path=FILE_PATH,
    log_column=LOG_COLUMN,
    max_systems=MAX_SYSTEMS,
    max_subsystems=MAX_SUBSYSTEMS,
    asset_context=ASSET_CONTEXT,
    is_free_text=IS_FREE_TEXT,
    sheet_name=SHEET_NAME,
)
# configure_result is automatically saved to {ARTIFACT_DIR}configure_labelling_pipeline.json
print(json.dumps(configure_result, indent=2))

## Step 2 — downsample  _(skip if `downsample_required` is False)_

Stratified sample to bring row count within the server limit.

In [ ]:
# ── Step 2 params — edit here ─────────────────────────────────────────────────
ASSET_COLUMN     = "asset_id"   # unique asset identifier — None if not available
VEHICLE_COLUMN   = None          # vehicle / fleet grouping — None if not available
TIMESTAMP_COLUMN = "date"        # event timestamp column — None if not available
DOWNSAMPLE_SIZE  = 5000          # target rows after sampling (max 5 000)

if configure_result.get("downsample_required"):
    downsample_result = session.downsample(
        sample_size=DOWNSAMPLE_SIZE,
        asset_column=ASSET_COLUMN,
        vehicle_column=VEHICLE_COLUMN,
        timestamp_column=TIMESTAMP_COLUMN,
        file_path=FILE_PATH,    # validates column names client-side before the API call
        sheet_name=SHEET_NAME,
    )
    # downsample_result is automatically saved to {ARTIFACT_DIR}downsample.json
    # downsampled.parquet is automatically downloaded to {ARTIFACT_DIR}downsampled.parquet
    print(json.dumps(downsample_result, indent=2))

    downsampled_path = downsample_result.get("downsampled_path")
    if downsampled_path:
        print(f"\nDownsampled rows saved to: {downsampled_path}")
        print("Inspect with:")
        print("  import pandas as pd")
        print(f"  pd.read_parquet('{downsampled_path}')")
else:
    print("Downsampling not required — skipped.")

In [ ]:
# ── Inspect the downsampled rows ──────────────────────────────────────────────
# Works immediately after downsample, or after kernel restart from the parquet file.
# Skipped automatically when downsampling was not required.
import os, pandas as pd

if configure_result.get("downsample_required"):
    downsampled_path = (
        downsample_result.get("downsampled_path")         # live result from above
        if "downsample_result" in dir()
        else f"{ARTIFACT_DIR}downsampled.parquet"         # kernel-restart fallback
    )
    if downsampled_path and os.path.exists(downsampled_path):
        downsampled_df = pd.read_parquet(downsampled_path)
        print(f"Sampled rows: {len(downsampled_df):,}  |  Columns: {list(downsampled_df.columns)}")
        display(downsampled_df.head())
    else:
        print(f"Parquet not found at {downsampled_path} — re-run Step 2 to regenerate.")
else:
    print("Downsampling was not required — full dataset is used directly.")

## Step 3 — extract_keywords

Scan log text and extract a vocabulary of domain-specific terms.

In [ ]:
# ── Step 3 params — edit here ─────────────────────────────────────────────────
EXTRACTION_METHOD = "deep"   # "fast" (default, recommended) | "deep" (slower, better for noisy text)

keywords_result = session.extract_keywords(extraction_method=EXTRACTION_METHOD, timeout=1200.0)
# keywords_result is automatically saved to {ARTIFACT_DIR}extract_keywords.json

keywords = keywords_result["keywords"]
print(f"Extracted {len(keywords)} keywords")
print(f"Raw count: {keywords_result['raw_keyword_count']}  →  Filtered: {keywords_result['filtered_keyword_count']}")
print(f"Linguistic filter removed: {keywords_result.get('linguistic_keywords_removed', 0)} terms")
print()
print(json.dumps(keywords_result, indent=2))


In [ ]:
# ── Optional: visualise keywords ──────────────────────────────────────────────
# All three forms below work equally:
#   (a) pass the result dict directly
#   (b) pass a file path to the saved artifact  ← useful after kernel restart
#   (c) pass the plain list of strings

# Word cloud (requires: pip install wordcloud matplotlib)
helpers.plot_keyword_cloud(keywords_result, title="Extracted Keywords", max_words=80)

# File path form — works even after kernel restart without re-running the step:
# helpers.plot_keyword_cloud(f"{ARTIFACT_DIR}extract_keywords.json")

In [ ]:
# ── Keyword frequency bar chart ───────────────────────────────────────────────
# Horizontal Plotly bar — highest-frequency terms at the top.
# Bar height = how many times each term appeared in the raw log text
# (from keyword_pool returned by extract_keywords — real counts, not just rank).
#
# Pass the result dict directly — frequencies are read from keyword_pool automatically.
# Also accepts: a file path to the artifact, a plain {keyword: freq} dict, or a list.

helpers.plot_top_keywords(
    keywords_result['keyword_pool'],
    top_n=50,
    title="Top 50 Extracted Keywords by Frequency",
)

# After kernel restart, use the saved artifact (frequencies preserved):
# helpers.plot_top_keywords(f"{ARTIFACT_DIR}extract_keywords.json", top_n=50)

In [ ]:
# ── Keyword frequency table ───────────────────────────────────────────────────
# Returns a DataFrame with columns ["Keyword", "Frequency"], sorted descending.
# Useful for exporting or inspecting exact counts.

df_keywords = helpers.get_top_keywords_table(keywords_result['keyword_pool'], top_n=50)
print(f"Top {len(df_keywords)} keywords:")
display(df_keywords)

# After kernel restart:
# df_keywords = helpers.get_top_keywords_table(f"{ARTIFACT_DIR}extract_keywords.json")

## Step 4 — generate_hierarchy

Send keywords to the LLM to build a two-level system / subsystem hierarchy.

In [ ]:
hierarchy_result = session.generate_hierarchy(
    timeout=1200,     # increase for large datasets or slow LLM
)
# hierarchy_result is automatically saved to {ARTIFACT_DIR}generate_hierarchy.json
print(json.dumps(hierarchy_result, indent=2))

# Optional: pass your own keywords to override the extracted ones
# hierarchy_result = session.generate_hierarchy(
#     keywords=["keyword1", "keyword2", ...],
#     timeout=1200,
# )

In [ ]:
# ── Optional: visualise the hierarchy ────────────────────────────────────────
# Pass the result dict, a file path, or a saved artifact path — all work.
# Note:- Colours :- "high":"green", "medium":"yellow", "low":"red"
        
helpers.plot_hierarchy(hierarchy_result, title="System / Subsystem Hierarchy")

# After kernel restart, use the saved artifact:
# helpers.plot_hierarchy(f"{ARTIFACT_DIR}generate_hierarchy.json")

In [ ]:
# ── View the hierarchy as a DataFrame ────────────────────────────────────────
import pandas as pd
hierarchy_df = pd.DataFrame(hierarchy_result["hierarchy"])

In [ ]:
hierarchy_df.iloc[:,0:4]

## Step 5 — update_hierarchy  _(optional, repeatable)_

Refine the LLM-generated hierarchy before locking it.

In [ ]:
# ── Export hierarchy to CSV for external editing ─────────────────────────────
# Saves the hierarchy as a CSV — open in Excel/Numbers, make your changes, save.
# Then load it back in the next cell and submit to update_hierarchy.

HIERARCHY_CSV = f"{ARTIFACT_DIR}hierarchy_draft.csv"
helpers.hierarchy_to_csv(hierarchy_result, HIERARCHY_CSV)
print(f"Hierarchy exported to: {HIERARCHY_CSV}")
print("Edit the file externally, then run the cell below to apply your changes.")

In [ ]:
# ── Load edited hierarchy CSV and submit ─────────────────────────────────────
# Run this after saving your edits to the CSV file above.

rows_from_csv = helpers.hierarchy_from_csv(HIERARCHY_CSV)
update_hierarchy_result = session.update_hierarchy(rows_from_csv)


In [ ]:
# View the updated hierarchy as a DataFrame again to confirm your changes took effect:
updated_hierarchy_result_df = pd.DataFrame(update_hierarchy_result["hierarchy"])
updated_hierarchy_result_df.iloc[:,0:4]

In [ ]:
# Visualise the updated hierarchy
helpers.plot_hierarchy(update_hierarchy_result, title="Updated System / Subsystem Hierarchy")

## Step 6 — generate_weights

Lock the hierarchy and generate criticality weights.

In [ ]:
weights_result = session.generate_weights()
# weights_result is automatically saved to {ARTIFACT_DIR}generate_weights.json
print(json.dumps(weights_result, indent=2))

In [ ]:
import pandas as pd
# Flat table: one row per subsystem with parent system weight alongside
rows = []
for s in weights_result["weights"]:
    for sub in s["subsystems"]:
        rows.append({
            "system":             s["system_name"],
            "system_weight":      s["weight"],
            "subsystem":          sub["subsystem_name"],
            "subsystem_weight":   sub["weight"],
        })

df_weights = pd.DataFrame(rows)
df_weights

In [ ]:
# ── Optional: visualise the generated weights ─────────────────────────────────
helpers.plot_weights(weights_result, title="System / Subsystem Weights")

# After kernel restart, load from artifact:
# helpers.plot_weights(f"{ARTIFACT_DIR}generate_weights.json")

## Step 7 — update_weights  _(optional, repeatable)_

Override the generated criticality weights.

In [ ]:
# ── Export weights to CSV for external editing ───────────────────────────────
# Saves weights as a flat CSV — one row per subsystem.
# Edit the 'system_weight' and 'subsystem_weight' columns in Excel, save, then load below.
# Do NOT rename system_name or subsystem_name columns.

WEIGHTS_CSV = f"{ARTIFACT_DIR}weights_draft.csv"
helpers.weights_to_csv(weights_result, WEIGHTS_CSV)
print(f"Weights exported to: {WEIGHTS_CSV}")
print("Edit the weight values externally, then run the cell below to apply.")

In [ ]:
# ── Load edited weights CSV and submit ───────────────────────────────────────
# Run after saving your edits to the CSV file above.
# update_weights auto-normalises: values you changed are locked, the rest are scaled.
# No need to manually ensure weights sum to 1.0 before passing.

systems_from_csv = helpers.weights_from_csv(WEIGHTS_CSV)
updated_result_weights = session.update_weights(
    systems_from_csv,
    original_systems=weights_result["weights"],  # locks changed values, protects names
)


In [ ]:
# View the updated weights as a DataFrame again to confirm your changes took effect:
import pandas as pd
# Flat table: one row per subsystem with parent system weight alongside
rows = []
for s in updated_result_weights["weights"]:
    for sub in s["subsystems"]:
        rows.append({
            "system":             s["system_name"],
            "system_weight":      s["weight"],
            "subsystem":          sub["subsystem_name"],
            "subsystem_weight":   sub["weight"],
        })

updated_df_weights = pd.DataFrame(rows)
updated_df_weights

In [ ]:
# Visualise the updated weights
helpers.plot_weights(updated_result_weights, title="Updated System / Subsystem Weights")

## Step 8 — run_classification  🔒 ONE-WAY DOOR

Classify every log entry and download the labelling zip. Cannot be repeated.

In [ ]:
OUTPUT_DIR = "outputs/fixed_log/"    # directory to save the zip into

classification_result = session.run_classification(OUTPUT_DIR, timeout=3600)
# classification_result is automatically saved to {ARTIFACT_DIR}run_classification.json

zip_path = classification_result["zip_path"]
print(f"Zip downloaded to: {zip_path}")
print("Session wiped automatically.")
print()
print("Next: open 02_risk_score.ipynb and pass this zip to import_model():")
print(f'  ZIP_PATH = "{zip_path}"')

## Classification Results

Distribution plots of the labelling output.

In [ ]:
# ── Chart 1: Stacked bar — log counts by system and subsystem ─────────────────
# Mirrors "create_system_subsystem_stacked_chart" in the Streamlit app.
import zipfile, pandas as pd

with zipfile.ZipFile(classification_result["zip_path"]) as zf:
    entry = next(n for n in zf.namelist() if "all_predictions_exploded" in n)
    with zf.open(entry) as f:
        exploded_df = pd.read_parquet(f)




In [ ]:
# Chart 1: Stacked bar — log counts by system and subsystem
helpers.plot_classification_stacked(
    exploded_df,
    title="Log Distribution — Systems & Subsystems",
)

In [ ]:
# ── Chart 2: Bar chart — total log count per system ──────────────────────────
# Uses exploded_df loaded in Chart 1 above.

helpers.plot_classification_by_system(
    exploded_df,
    title="Log Distribution per System",
)

In [ ]:
# ── Chart 3: Horizontal bar — top N subsystems by log count ──────────────────
# Uses exploded_df loaded in Chart 1 above.

helpers.plot_classification_by_subsystem(
    exploded_df,
    top_n=20,
    title="Top 20 Subsystems by Log Count",
)